# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alikadirguzel/flyrankinternship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

Lane **2 — Refresh / Content Opportunity Scoring** (locked this week). One transparent rule, checked against two signals before it is encoded. This queue is the baseline Week 5 has to beat.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `building-baselines` plus `flyrank/flyrank-data`.

## 1. My rule and its reason codes

**Lane (locked):** Refresh / Content Opportunity Scoring.

**The rule in plain words.** A page is worth an editor's hour if it still shows up in search *and* it has gone a long time without an update. Rank those pages by how stale they are, so the oldest visible pages come first. Volume is a gate (skip the tiny URLs), not the ranker.

That is the same idea as FlyRank's refresh flags (staleness) plus the volume floor behind quick-win. I check both signals before coding anything. A negative here is useful: it stops me from shipping a product cutoff that this slice does not support.

**Candidate gates (written before looking at the buckets):**
- visible: `impressions_90d >= 500`
- stale: `days_since_last_update >= 90` (the live flag uses 180; this slice may not)

**What the rule is allowed to emit**
- One reason code when it fires: `stale_visible_page`
- One action when it fires: `refresh`
- Default when a gate fails: reason `not_flagged`, action `monitor` — that is the zero-score floor, not a second rule

**Two signals this rule leans on** (at least one is flag-linked):

| # | Signal | FlyRank flag it sits behind | Claim I am testing |
|---|---|---|---|
| 1 | Staleness (`days_since_last_update` / `freshness_tier`) | refresh flags | Older-update pages show a higher decline rate |
| 2 | Volume (`impressions_90d` / `impression_tier`) | quick-win | Higher-volume pages are the ones worth acting on / more often declining |

Proxy used only as a *check*, never as a score input: `is_declining_label = (trend_direction == "down")`. Base rate on this 30,000-row snapshot is about 54%. `trend_direction` / `trend_pct` stay out of the rule.

Verdict vocabulary: **CONFIRMED / OPPOSITE / MIXED / FALSE**. Each test is a bucket table with `n` in every row. No verdict from a cell under ~50 rows.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

candidates = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("flyrankinternship/data/raw/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    if not Path("flyrankinternship").exists():
        get_ipython().system("git clone https://github.com/alikadirguzel/flyrankinternship.git")
    data_path = Path("flyrankinternship/data/raw/content_refresh_anonymized.csv")

# data_path is <repo>/data/raw/content_refresh_anonymized.csv → parents[2] is <repo>
REPO_ROOT = data_path.resolve().parents[2]
OUT_DIR = REPO_ROOT / "work" / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
if not (REPO_ROOT / "data" / "raw").exists():
    raise SystemExit(f"repo-root lookup failed: {REPO_ROOT}")

df = pd.read_csv(data_path)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

n_rows = len(df)
base_rate = float(df["is_declining_label"].mean())
print(f"loaded: {data_path}")
print(f"rows x cols: {df.shape[0]:,} x {df.shape[1]}")
print(f"repo root: {REPO_ROOT}")
print(f"proxy: is_declining_label = (trend_direction == 'down')")
print(f"base rate (random ranking's expected Precision@K): {base_rate:.3f}  ({int(df['is_declining_label'].sum()):,} / {n_rows:,})")
print("score inputs will be days_since_last_update and impressions_90d only")
print("excluded from the score: trend_direction, trend_pct, *_last_30d, *_prev_30d")
print()

FLOOR_N = 50


def bucket_table(frame, by, label_col="is_declining_label"):
    g = (
        frame.groupby(by, observed=True)
        .agg(
            n=(label_col, "size"),
            n_declining=(label_col, "sum"),
            decline_rate=(label_col, "mean"),
            median_impressions=("impressions_90d", "median"),
            median_days_since_update=("days_since_last_update", "median"),
        )
        .reset_index()
    )
    g["decline_rate"] = g["decline_rate"].round(4)
    g["vs_base_pp"] = ((g["decline_rate"] - base_rate) * 100).round(1)
    g["n_ok"] = np.where(g["n"] >= FLOOR_N, "ok", "below_floor")
    return g


# ---------------------------------------------------------------------------
# Signal 1 — staleness (behind the refresh flags)
# Claim: pages left untouched longer have a higher decline rate.
# ---------------------------------------------------------------------------
print("=" * 72)
print("SIGNAL 1 — staleness (behind FlyRank refresh flags)")
print("claim: older days_since_last_update -> higher decline rate")
print("=" * 72)

fresh_order = ["0-30", "31-90", "91-180", "181+"]
df["freshness_tier"] = pd.Categorical(df["freshness_tier"], categories=fresh_order, ordered=True)

stale_all = bucket_table(df, "freshness_tier")
print()
print("freshness_tier vs decline proxy — ALL pages")
display(stale_all)

visible = df[df["impressions_90d"] >= 500].copy()
stale_vis = bucket_table(visible, "freshness_tier")
print()
print("freshness_tier vs decline proxy — VISIBLE pages only (impressions_90d >= 500)")
display(stale_vis)

n_flag_180_vis = int(((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum())
n_181_vis = int(stale_vis.loc[stale_vis["freshness_tier"] == "181+", "n"].sum()) if "181+" in set(stale_vis["freshness_tier"].astype(str)) else 0
print()
print(f"product-style refresh flag on this slice: days_since_last_update >= 180 AND impressions_90d >= 500")
print(f"pages that flag fires on: {n_flag_180_vis:,}  (sample-size floor is {FLOOR_N} — this cell is below it)")

# One-word verdict: MIXED. Moderate staleness (91-180) is above base rate;
# the 181+ product cutoff is not readable (n=17 when visible; overall 181+ is
# actually BELOW base rate and has almost no traffic).
verdict_staleness = "MIXED"
print()
print(f"VERDICT signal 1 (staleness / refresh flags): {verdict_staleness}")
print("reading: 91-180 is the bucket that actually looks worse (61% declining, n=9,171).")
print("reading: 181+ overall is 47% declining with median 16 impressions — the 180-day")
print("         product cutoff without a volume floor is the wrong gate on this snapshot.")
print("reading: 181+ AND visible is 16/17 declining, but n=17 is below the floor, so that")
print("         intersection is not a confirmed flag. It just cannot be the whole queue.")
print()

# ---------------------------------------------------------------------------
# Signal 2 — volume (behind quick-win)
# Claim: more impressions => more worth acting on / more often declining.
# ---------------------------------------------------------------------------
print("=" * 72)
print("SIGNAL 2 — volume (behind FlyRank quick-win)")
print("claim: higher impressions_90d -> higher decline rate / clearer opportunity")
print("=" * 72)

imp_order = ["low", "moderate", "good", "excellent"]
df["impression_tier"] = pd.Categorical(df["impression_tier"], categories=imp_order, ordered=True)
vol_all = bucket_table(df, "impression_tier")
print()
print("impression_tier vs decline proxy — ALL pages")
display(vol_all)

df["imp_bucket"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 50, 100, 500, 3_000, 30_000, 10_000_000],
    labels=["1-50", "51-100", "101-500", "501-3k", "3k-30k", "30k+"],
)
vol_cut = bucket_table(df, "imp_bucket")
print()
print("impressions_90d cut vs decline proxy (same claim, finer buckets, n in every row)")
display(vol_cut)

# Naive rank-by-volume, to show why volume must not be the score itself.
order_imp = np.argsort(-df["impressions_90d"].to_numpy(), kind="mergesort")
p50_volume_only = float(df["is_declining_label"].to_numpy()[order_imp[:50]].mean())
print()
print(f"Precision@50 if I rank the whole inventory by impressions_90d: {p50_volume_only:.3f}")
print(f"vs base rate {base_rate:.3f}  ({p50_volume_only - base_rate:+.3f})")

verdict_volume = "MIXED"
print()
print(f"VERDICT signal 2 (volume / quick-win): {verdict_volume}")
print("reading: skipping the low-volume tail is supported — `low` is 45% declining and is")
print("         full of new/flat pages. A 500-impression gate is a reasonable floor.")
print("reading: 'more volume is more opportunity' is OPPOSITE at the top. `excellent`")
print("         (30k+ impressions) is 46% declining, below base rate, and ranking by")
print("         impressions alone loses to random at Precision@50.")
print("reading: that saved the rule. Volume stays a GATE. It does not multiply the score.")


loaded: data\raw\content_refresh_anonymized.csv
rows x cols: 30,000 x 45
repo root: C:\Users\Ali Kadir Güzel\OneDrive\Masaüstü\şirket\flyrank\flyrankinternship
proxy: is_declining_label = (trend_direction == 'down')
base rate (random ranking's expected Precision@K): 0.542  (16,262 / 30,000)
score inputs will be days_since_last_update and impressions_90d only
excluded from the score: trend_direction, trend_pct, *_last_30d, *_prev_30d

SIGNAL 1 — staleness (behind FlyRank refresh flags)
claim: older days_since_last_update -> higher decline rate

freshness_tier vs decline proxy — ALL pages


,freshness_tier,n,n_declining,decline_rate,median_impressions,median_days_since_update,vs_base_pp,n_ok
0,0-30,20480,10473,0.5114,470.0,20.0,-3.1,ok
1,31-90,175,103,0.5886,510.0,41.0,4.7,ok
2,91-180,9171,5604,0.6111,1692.0,104.0,6.9,ok
3,181+,174,82,0.4713,15.5,211.0,-7.1,ok



freshness_tier vs decline proxy — VISIBLE pages only (impressions_90d >= 500)


,freshness_tier,n,n_declining,decline_rate,median_impressions,median_days_since_update,vs_base_pp,n_ok
0,0-30,10063,5862,0.5825,2703.0,20.0,4.0,ok
1,31-90,88,46,0.5227,1314.5,41.0,-1.9,ok
2,91-180,6558,4037,0.6156,3434.5,104.0,7.4,ok
3,181+,17,16,0.9412,4429.0,194.0,39.9,below_floor



product-style refresh flag on this slice: days_since_last_update >= 180 AND impressions_90d >= 500
pages that flag fires on: 17  (sample-size floor is 50 — this cell is below it)

VERDICT signal 1 (staleness / refresh flags): MIXED
reading: 91-180 is the bucket that actually looks worse (61% declining, n=9,171).
reading: 181+ overall is 47% declining with median 16 impressions — the 180-day
         product cutoff without a volume floor is the wrong gate on this snapshot.
reading: 181+ AND visible is 16/17 declining, but n=17 is below the floor, so that
         intersection is not a confirmed flag. It just cannot be the whole queue.

SIGNAL 2 — volume (behind FlyRank quick-win)
claim: higher impressions_90d -> higher decline rate / clearer opportunity

impression_tier vs decline proxy — ALL pages


,impression_tier,n,n_declining,decline_rate,median_impressions,median_days_since_update,vs_base_pp,n_ok
0,low,11248,5106,0.4539,31.0,20.0,-8.8,ok
1,moderate,10469,6435,0.6147,998.0,22.0,7.3,ok
2,good,7205,4223,0.5861,7249.0,22.0,4.4,ok
3,excellent,1078,498,0.4620,48675.0,25.0,-8.0,ok



impressions_90d cut vs decline proxy (same claim, finer buckets, n in every row)


,imp_bucket,n,n_declining,decline_rate,median_impressions,median_days_since_update,vs_base_pp,n_ok
0,1-50,6528,2247,0.3442,6.0,20.0,-19.8,ok
1,51-100,1478,869,0.5880,73.0,20.0,4.6,ok
2,101-500,5279,3190,0.6043,253.0,22.0,6.2,ok
3,501-3k,8432,5235,0.6208,1231.0,22.0,7.9,ok
4,3k-30k,7205,4223,0.5861,7249.0,22.0,4.4,ok
5,30k+,1078,498,0.4620,48675.0,25.0,-8.0,ok



Precision@50 if I rank the whole inventory by impressions_90d: 0.420
vs base rate 0.542  (-0.122)

VERDICT signal 2 (volume / quick-win): MIXED
reading: skipping the low-volume tail is supported — `low` is 45% declining and is
         full of new/flat pages. A 500-impression gate is a reasonable floor.
reading: 'more volume is more opportunity' is OPPOSITE at the top. `excellent`
         (30k+ impressions) is 46% declining, below base rate, and ranking by
         impressions alone loses to random at Precision@50.
reading: that saved the rule. Volume stays a GATE. It does not multiply the score.


## 2. Build the ranked queue (writes the CSV)

The two verdicts changed the rule:

- Staleness is **MIXED**, so I do **not** copy the 180-day product cutoff. On this snapshot the decline rate rises in `91-180`, and `181+` without a volume floor is a low-traffic dead end. Gate: `days_since_last_update >= 90`.
- Volume is **MIXED**, so I do **not** rank by raw impressions. Gate: `impressions_90d >= 500`. Rank by days since update.

**Encoded rule (no fitted weights):**

```text
stale   = (days_since_last_update >= 90)
visible = (impressions_90d >= 500)
score   = stale * visible * days_since_last_update
```

- reason code: `stale_visible_page` when score > 0, else `not_flagged`
- action label: `refresh` when score > 0, else `monitor`
- ties (this slice has a huge 104-day plateau): higher `impressions_90d`, then `content_id` — operational, not a trained weight

Inputs are trailing-90-day observables only. No `trend_*`, no last-30 vs prev-30, no product flags (they are not in this file).

The notebook writes `work/outputs/baseline_action_score.csv` (gitignored on purpose) and `work/outputs/baseline_metrics.json` (the run receipt).


In [2]:
import json

STALE_DAYS = 90
VISIBLE_IMP = 500

stale = (df["days_since_last_update"] >= STALE_DAYS).astype(int)
visible = (df["impressions_90d"] >= VISIBLE_IMP).astype(int)

# Transparent score: gates * days. Volume is a 0/1 gate, not a multiplier.
df["score"] = stale * visible * df["days_since_last_update"]
df["reason_code"] = np.where(df["score"] > 0, "stale_visible_page", "not_flagged")
df["action_label"] = np.where(df["score"] > 0, "refresh", "monitor")

queue = df.sort_values(
    ["score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
    kind="mergesort",
).reset_index(drop=True)
queue["rank"] = np.arange(1, len(queue) + 1)

export_cols = [
    "rank",
    "content_id",
    "client_id",
    "score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "content_age_days",
    "freshness_tier",
    "impression_tier",
    "position_tier",
    "content_type",
    "is_declining_label",
    "trend_direction",
]
# trend_direction / is_declining_label are EVALUATION context, not score inputs.
queue_out = queue[export_cols].copy()

csv_path = OUT_DIR / "baseline_action_score.csv"
queue_out.to_csv(csv_path, index=False)

n_flagged = int((queue["score"] > 0).sum())
precision_flagged = float(queue.loc[queue["score"] > 0, "is_declining_label"].mean())


def precision_at_k(frame, k):
    top = frame.head(k)
    return float(top["is_declining_label"].mean()), int(top["is_declining_label"].sum()), k


p10, h10, _ = precision_at_k(queue, 10)
p20, h20, _ = precision_at_k(queue, 20)
p50, h50, _ = precision_at_k(queue, 50)

print("rule in code:")
print(f"  stale   = days_since_last_update >= {STALE_DAYS}")
print(f"  visible = impressions_90d >= {VISIBLE_IMP}")
print("  score   = stale * visible * days_since_last_update")
print("  reason  = stale_visible_page  (else not_flagged)")
print("  action  = refresh             (else monitor)")
print()
print(f"flagged (score > 0): {n_flagged:,} / {len(queue):,}")
print(f"reason-code mix:\n{queue['reason_code'].value_counts().to_string()}")
print()
print(f"action-label mix:\n{queue['action_label'].value_counts().to_string()}")
print()
print("Precision@K on this snapshot (in-sample peek — Week 5 should beat this on a client-holdout):")
metrics_tbl = pd.DataFrame(
    [
        {"ranking": "this rule", "K": 10, "Precision@K": round(p10, 3), "hits": h10, "vs_base": round(p10 - base_rate, 3)},
        {"ranking": "this rule", "K": 20, "Precision@K": round(p20, 3), "hits": h20, "vs_base": round(p20 - base_rate, 3)},
        {"ranking": "this rule", "K": 50, "Precision@K": round(p50, 3), "hits": h50, "vs_base": round(p50 - base_rate, 3)},
        {"ranking": "impressions_90d only", "K": 50, "Precision@K": round(p50_volume_only, 3), "hits": int(p50_volume_only * 50), "vs_base": round(p50_volume_only - base_rate, 3)},
        {"ranking": "random / base rate", "K": "any", "Precision@K": round(base_rate, 3), "hits": None, "vs_base": 0.0},
    ]
)
display(metrics_tbl)
print()
print(f"precision of the WHOLE flagged set (n={n_flagged:,}): {precision_flagged:.3f}")
print("reading: Precision@50 looks strong because the extreme stale tail is tiny and")
print("         happens to be declining. The 6,575-page flagged set is a modest lift")
print("         over base rate (~0.62 vs 0.54). Week 5 should beat the ranking, not")
print("         a 17-row 181+ coincidence.")
print()
print("top of the written queue:")
display(
    queue_out.head(10)[
        [
            "rank",
            "content_id",
            "score",
            "reason_code",
            "action_label",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "trend_direction",
        ]
    ]
)

receipt = {
    "lane": "refresh_content_opportunity_scoring",
    "rule_plain": "visible and stale; rank by days since last update",
    "score_formula": "stale * visible * days_since_last_update",
    "thresholds": {
        "days_since_last_update_min": STALE_DAYS,
        "impressions_90d_min": VISIBLE_IMP,
    },
    "reason_code_when_flagged": "stale_visible_page",
    "action_label_when_flagged": "refresh",
    "n_rows": int(len(queue)),
    "n_flagged": n_flagged,
    "base_rate": round(base_rate, 6),
    "precision_flagged": round(precision_flagged, 6),
    "precision_at_10": round(p10, 6),
    "precision_at_20": round(p20, 6),
    "precision_at_50": round(p50, 6),
    "precision_at_50_volume_only": round(p50_volume_only, 6),
    "signal_verdicts": {
        "staleness_refresh_flags": verdict_staleness,
        "volume_quick_win": verdict_volume,
    },
    "features_used_in_score": ["days_since_last_update", "impressions_90d"],
    "excluded_from_score": [
        "trend_direction",
        "trend_pct",
        "impressions_last_30d",
        "impressions_prev_30d",
        "clicks_last_30d",
        "clicks_prev_30d",
        "sessions_last_30d",
        "sessions_prev_30d",
    ],
    "csv_path": "work/outputs/baseline_action_score.csv",
}

json_path = OUT_DIR / "baseline_metrics.json"
json_path.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print()
print(f"wrote queue:   {csv_path}")
print(f"wrote receipt: {json_path}")
print(f"csv rows: {len(queue_out):,}  (gitignored; regenerate by running this notebook)")


rule in code:
  stale   = days_since_last_update >= 90
  visible = impressions_90d >= 500
  score   = stale * visible * days_since_last_update
  reason  = stale_visible_page  (else not_flagged)
  action  = refresh             (else monitor)

flagged (score > 0): 6,575 / 30,000
reason-code mix:
reason_code
not_flagged           23425
stale_visible_page     6575

action-label mix:
action_label
monitor    23425
refresh     6575

Precision@K on this snapshot (in-sample peek — Week 5 should beat this on a client-holdout):


,ranking,K,Precision@K,hits,vs_base
0,this rule,10,0.900,9.0,0.358
1,this rule,20,0.800,16.0,0.258
2,this rule,50,0.860,43.0,0.318
3,impressions_90d only,50,0.420,21.0,-0.122
4,random / base rate,any,0.542,NaN,0.000



precision of the WHOLE flagged set (n=6,575): 0.616
reading: Precision@50 looks strong because the extreme stale tail is tiny and
         happens to be declining. The 6,575-page flagged set is a modest lift
         over base rate (~0.62 vs 0.54). Week 5 should beat the ranking, not
         a 17-row 181+ coincidence.

top of the written queue:


,rank,content_id,score,reason_code,action_label,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction
0,1,content_7f116ae1f6f5,301,stale_visible_page,refresh,301,954,9.0,0.42,down
1,2,content_72496874f806,301,stale_visible_page,refresh,301,821,5.8,0.24,down
2,3,content_cf56e2e2e282,194,stale_visible_page,refresh,194,61678,19.7,0.15,down
3,4,content_7368877ea310,194,stale_visible_page,refresh,194,59472,24.8,0.13,down
4,5,content_1bfaa38ff26c,194,stale_visible_page,refresh,194,25715,22.2,0.23,down
5,6,content_5feee3994adb,194,stale_visible_page,refresh,194,7812,39.0,0.01,down
6,7,content_b16bd7307b39,194,stale_visible_page,refresh,194,4590,31.0,0.00,down
7,8,content_fe16a55cd13d,194,stale_visible_page,refresh,194,4556,16.4,0.33,down
8,9,content_ecb6215e79fd,194,stale_visible_page,refresh,194,4429,25.3,0.38,down
9,10,content_bdbec75c1148,194,stale_visible_page,refresh,194,1316,21.8,0.15,stable



wrote queue:   C:\Users\Ali Kadir Güzel\OneDrive\Masaüstü\şirket\flyrank\flyrankinternship\work\outputs\baseline_action_score.csv
wrote receipt: C:\Users\Ali Kadir Güzel\OneDrive\Masaüstü\şirket\flyrank\flyrankinternship\work\outputs\baseline_metrics.json
csv rows: 30,000  (gitignored; regenerate by running this notebook)


## 3. Top-20 review

The assignment asks for ten rows; the skeleton asks for twenty. Each line is: action, why it is here, and what would make it wrong. Rank comes from the CSV sort (`score`, then impressions, then `content_id`). I have not looked at `trend_direction` to *place* them — I use it only to say whether the proxy later agrees.

**Top 10 (required)**

1. `content_7f116ae1f6f5` — **refresh** — 301 days, 954 impressions, page 1, CTR 0.42. Why: oldest visible page the gates allow. Wrong if the CMS "last update" date is stale while the live copy was edited elsewhere, or if 4 clicks / 6 sessions means this URL is not actually catching the query.

2. `content_72496874f806` — **refresh** — 301 days, 821 impressions, page 1, CTR 0.24. Why: same oldest-visible gate. Wrong if position 5.8 is branded/navigational demand a rewrite will not move, or if 2 clicks is noise around the 500-impression floor.

3. `content_cf56e2e2e282` — **refresh** — 194 days, 61,678 impressions, position ~20. Why: still very stale, and now actually high-visibility. Wrong if this traffic is one seasonal spike already over, or if a 19.7 average position is a mix of one head query and a long tail a refresh cannot lift together.

4. `content_7368877ea310` — **refresh** — 194 days, 59,472 impressions, page 3–5. Why: same stale+visible pattern, huge volume. Wrong if page 3–5 is a linking/authority problem rather than a copy-age problem — an editor hour on the article would not be the bottleneck.

5. `content_1bfaa38ff26c` — **refresh** — 194 days, 25,715 impressions, page 3–5. Why: stale, still visible. Wrong if the page is a comparison/utility URL whose SERP lost a featured snippet; updating paragraphs would not restore the clicks.

6. `content_5feee3994adb` — **refresh** — 194 days, 7,812 impressions, position 39, CTR 0.01, 1 click. Why: passes both gates. Wrong if position 39 means the page is not in the game — refresh will not create demand that search is not sending.

7. `content_b16bd7307b39` — **refresh** — 194 days, 4,590 impressions, position 31, **0 clicks**. Why: stale and above the volume floor. Wrong if zero clicks in 90 days means impressions are impressions-of-a-URL-in-a-bundle, not a page anyone is choosing; the action should be `monitor` or prune, not refresh.

8. `content_fe16a55cd13d` — **refresh** — 194 days, 4,556 impressions, striking distance, 15 clicks. Why: stale, some actual clicks. Wrong if 42 sessions with a 16.4 position is already a healthy page whose date is just old.

9. `content_ecb6215e79fd` — **refresh** — 194 days, 4,429 impressions, page 3–5. Why: same client-batch as #3–#8. Wrong if this is one site that stopped publishing on a calendar, not 8 independent decay stories — one client's freeze inflates the top of the queue.

10. `content_bdbec75c1148` — **refresh** — 194 days, 1,316 impressions, page 3–5, **stable** on the proxy. Why: the rule cannot see momentum, only age + volume. Wrong: it already is, if the job is "catch declining pages." This is the first clear false alarm in the top ten.

**11–20 (stretch)**

11. `content_77d4d5930e5e` — **refresh** — 194 days, 828 impressions, striking, 2 clicks. Why: last of the 194-day visible cluster. Wrong if 828 impressions is too close to the floor for a confident CTR/position read.

12. `content_0a91db491d14` — **refresh** — 193 days, 13,299 impressions, striking, 65 clicks. Why: next-oldest visible batch, real click volume. Wrong if striking-distance + 0.49 CTR is already fine and the date is cosmetic.

13. `content_c2d929d83eaa` — **refresh** — 193 days, 7,558 impressions, striking. Why: same batch. Wrong if 15 clicks in 90 days is not enough on-page evidence to spend an hour.

14. `content_928af3e22c80` — **refresh** — 193 days, 1,697 impressions, striking, 2 clicks. Why: stale+visible. Wrong if this is a thin-click URL whose impressions come from a broad query it cannot win.

15. `content_e3ff1b093148` — **refresh** — 183 days, 1,408 impressions, page 1, CTR 0.28. Why: still past 180 days and on page 1. Wrong if page-1 with 4 clicks is a low-demand query that is already in the right place.

16. `content_6226ee6adc91` — **refresh** — 183 days, 545 impressions (barely above the gate), 1 click. Why: clears both cutoffs. Wrong if the 500 floor is doing all the work — one more quiet week and this page would not belong on any list.

17. `content_074ba6ead17b` — **refresh** — 183 days, 533 impressions, position 48, 0 clicks. Why: same barely-visible stale gate. Wrong if a deep URL with no clicks is a prune/noindex candidate, not a refresh.

18. `content_cb7e312f5d32` — **refresh** — 151 days, 21,272 impressions, **CTR 2.45, 522 clicks, trend up**. Why: age + volume only. Wrong: already is. This page is gathering clicks. Refreshing it now is the expensive false positive.

19. `content_2f38c656fc95` — **refresh** — 151 days, 1,830 impressions, deep pack, trend up. Why: stale enough to pass. Wrong if a position-52 page that is already rising needs monitoring, not a rewrite.

20. `content_6c56c0675297` — **refresh** — 151 days, 1,790 impressions, page 3–5, **stable**. Why: same 151-day cluster. Wrong if stable + low clicks is "leave it"; the rule has no healthy-page off-ramp besides the two gates.


In [3]:
print("top 20 of the written queue — action / reason / observables used to review")
top20 = queue_out.head(20).copy()
display(
    top20[
        [
            "rank",
            "content_id",
            "action_label",
            "reason_code",
            "score",
            "days_since_last_update",
            "impressions_90d",
            "avg_position",
            "ctr",
            "clicks_90d",
            "sessions_90d",
            "position_tier",
            "trend_direction",
            "is_declining_label",
        ]
    ]
)

print()
print("one-line audit keys for the markdown review above:")
for _, row in top20.iterrows():
    print(
        f"{int(row['rank']):2d}. {row['content_id']}  {row['action_label']:8s}  "
        f"{row['reason_code']:20s}  days={int(row['days_since_last_update']):3d}  "
        f"imp={int(row['impressions_90d']):6d}  pos={row['avg_position']}  "
        f"ctr={row['ctr']}  clicks={int(row['clicks_90d'])}  "
        f"proxy={row['trend_direction']}"
    )

print()
print(f"top-10 proxy hits: {int(top20.head(10)['is_declining_label'].sum())} / 10")
print(f"top-20 proxy hits: {int(top20['is_declining_label'].sum())} / 20")
print("client mix in top 20 (batch risk):")
print(queue.head(20)["client_id"].value_counts().to_string())


top 20 of the written queue — action / reason / observables used to review


,rank,content_id,action_label,reason_code,score,days_since_last_update,impressions_90d,avg_position,ctr,clicks_90d,sessions_90d,position_tier,trend_direction,is_declining_label
0,1,content_7f116ae1f6f5,refresh,stale_visible_page,301,301,954,9.0,0.42,4,6,page_1,down,1
1,2,content_72496874f806,refresh,stale_visible_page,301,301,821,5.8,0.24,2,6,page_1,down,1
2,3,content_cf56e2e2e282,refresh,stale_visible_page,194,194,61678,19.7,0.15,94,119,striking,down,1
3,4,content_7368877ea310,refresh,stale_visible_page,194,194,59472,24.8,0.13,77,82,page_3_5,down,1
4,5,content_1bfaa38ff26c,refresh,stale_visible_page,194,194,25715,22.2,0.23,60,80,page_3_5,down,1
5,6,content_5feee3994adb,refresh,stale_visible_page,194,194,7812,39.0,0.01,1,5,page_3_5,down,1
6,7,content_b16bd7307b39,refresh,stale_visible_page,194,194,4590,31.0,0.00,0,4,page_3_5,down,1
7,8,content_fe16a55cd13d,refresh,stale_visible_page,194,194,4556,16.4,0.33,15,42,striking,down,1
8,9,content_ecb6215e79fd,refresh,stale_visible_page,194,194,4429,25.3,0.38,17,12,page_3_5,down,1
9,10,content_bdbec75c1148,refresh,stale_visible_page,194,194,1316,21.8,0.15,2,6,page_3_5,stable,0



one-line audit keys for the markdown review above:
 1. content_7f116ae1f6f5  refresh   stale_visible_page    days=301  imp=   954  pos=9.0  ctr=0.42  clicks=4  proxy=down
 2. content_72496874f806  refresh   stale_visible_page    days=301  imp=   821  pos=5.8  ctr=0.24  clicks=2  proxy=down
 3. content_cf56e2e2e282  refresh   stale_visible_page    days=194  imp= 61678  pos=19.7  ctr=0.15  clicks=94  proxy=down
 4. content_7368877ea310  refresh   stale_visible_page    days=194  imp= 59472  pos=24.8  ctr=0.13  clicks=77  proxy=down
 5. content_1bfaa38ff26c  refresh   stale_visible_page    days=194  imp= 25715  pos=22.2  ctr=0.23  clicks=60  proxy=down
 6. content_5feee3994adb  refresh   stale_visible_page    days=194  imp=  7812  pos=39.0  ctr=0.01  clicks=1  proxy=down
 7. content_b16bd7307b39  refresh   stale_visible_page    days=194  imp=  4590  pos=31.0  ctr=0.0  clicks=0  proxy=down
 8. content_fe16a55cd13d  refresh   stale_visible_page    days=194  imp=  4556  pos=16.4  ctr=0.33  c

## 4. Weak picks + leakage check

**Weak picks I would not send an editor without a second look**

- **#10** (`content_bdbec75c1148`) — stable on the proxy. Age + volume with no off-ramp for a page that is holding still.
- **#18** (`content_cb7e312f5d32`) — 522 clicks, CTR 2.45 (x100, so 2.45%), trend up. The worst miss: a working page in the refresh queue.
- **#6, #7, #16, #17** — stale enough to pass, but 0–1 clicks and/or positions 31–48. The 500-impression floor still lets through URLs that look like "seen in search," not "chosen in search."
- **Client batch** — 8 of the top 11 share one `client_id` at 193–194 days. That is one site pausing updates, not 8 independent opportunities. A later model needs a client-grouped split or the ranker will overfit one calendar freeze.
- **104-day plateau** — 6,450 of 6,575 flagged pages sit at exactly 104 days. After the short 181+ tail, the score is almost a tie, and impressions become the only sort key. That is a limitation of this snapshot, not a deep freshness signal.

**What the rule cannot see (on purpose)**
- CTR vs position (the CTR-fix flag). I did not encode it; volume and staleness were the two tests this refresh rule leans on.
- Word count / thin content.
- Engagement rate.

**Leakage check.** The score uses `days_since_last_update` and `impressions_90d` only. It does not use `trend_direction`, `trend_pct`, last-30 vs prev-30 windows, or any FlyRank product flag (`health_score`, `needs_ctr_fix`, `is_quick_win` — those columns are not in this file). The proxy label is attached after ranking, for Precision@K and for this review. That is evaluation, not an input.


In [4]:
print("LEAKAGE CHECK — columns that entered the score")
print("  used:     days_since_last_update, impressions_90d")
print("  not used: trend_direction, trend_pct, any *_last_30d / *_prev_30d")
print("  not used: product flags (they are not columns in this snapshot)")
print()

forbidden = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "impressions_prev_30d",
    "clicks_last_30d",
    "clicks_prev_30d",
    "sessions_last_30d",
    "sessions_prev_30d",
]
present_forbidden = [c for c in forbidden if c in df.columns]
print("forbidden columns present in the file (evaluation/context only):")
print(" ", present_forbidden)
print()

# Rebuild the score from the two allowed columns and prove it matches.
rebuilt = (
    (df["days_since_last_update"] >= STALE_DAYS).astype(int)
    * (df["impressions_90d"] >= VISIBLE_IMP).astype(int)
    * df["days_since_last_update"]
)
print(f"rebuilt score equals df['score'] on every row: {bool((rebuilt == df['score']).all())}")
print()

print("weak-pick slice inside the top 20 (proxy not down, or 0-1 clicks):")
weak = top20[
    (top20["is_declining_label"] == 0)
    | (top20["clicks_90d"] <= 1)
    | (top20["avg_position"] > 30)
][
    [
        "rank",
        "content_id",
        "action_label",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "clicks_90d",
        "ctr",
        "trend_direction",
    ]
]
display(weak)

print()
print("104-day plateau among flagged rows:")
flagged = queue[queue["score"] > 0]
print(f"  flagged n = {len(flagged):,}")
print(f"  exactly 104 days = {int((flagged['days_since_last_update'] == 104).sum()):,}")
print()
print("client concentration, top 20:")
print(queue.head(20).groupby("client_id").size().sort_values(ascending=False).head(5).to_string())
print()
print("no product-flag columns in this file:")
productish = [c for c in df.columns if any(k in c.lower() for k in ["health_score", "priority_score", "needs_ctr", "quick_win", "action_type", "refresh_flag"])]
print(" ", productish if productish else "(none)")
print()
print("CSV exists and has 30,000 ranked rows:")
written = pd.read_csv(csv_path)
print(f"  {csv_path.name}: {len(written):,} rows, rank 1..{int(written['rank'].max())}, unique ranks={written['rank'].nunique():,}")
print(f"  reason codes in file: {sorted(written['reason_code'].unique().tolist())}")
print(f"  actions in file:      {sorted(written['action_label'].unique().tolist())}")


LEAKAGE CHECK — columns that entered the score
  used:     days_since_last_update, impressions_90d
  not used: trend_direction, trend_pct, any *_last_30d / *_prev_30d
  not used: product flags (they are not columns in this snapshot)

forbidden columns present in the file (evaluation/context only):
  ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']

rebuilt score equals df['score'] on every row: True

weak-pick slice inside the top 20 (proxy not down, or 0-1 clicks):


,rank,content_id,action_label,days_since_last_update,impressions_90d,avg_position,clicks_90d,ctr,trend_direction
5,6,content_5feee3994adb,refresh,194,7812,39.0,1,0.01,down
6,7,content_b16bd7307b39,refresh,194,4590,31.0,0,0.00,down
9,10,content_bdbec75c1148,refresh,194,1316,21.8,2,0.15,stable
15,16,content_6226ee6adc91,refresh,183,545,17.8,1,0.18,down
16,17,content_074ba6ead17b,refresh,183,533,48.0,0,0.00,down
17,18,content_cb7e312f5d32,refresh,151,21272,12.6,522,2.45,up
18,19,content_2f38c656fc95,refresh,151,1830,52.5,6,0.33,up
19,20,content_6c56c0675297,refresh,151,1790,46.0,7,0.39,stable



104-day plateau among flagged rows:
  flagged n = 6,575
  exactly 104 days = 6,450

client concentration, top 20:
client_id
client_7f2253d7e2    12
client_9f14025af0     3
client_d029fa3a95     3
client_4ec9599fc2     1
client_9400f1b21c     1

no product-flag columns in this file:
  (none)

CSV exists and has 30,000 ranked rows:
  baseline_action_score.csv: 30,000 rows, rank 1..30000, unique ranks=30,000
  reason codes in file: ['not_flagged', 'stale_visible_page']
  actions in file:      ['monitor', 'refresh']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Lane lock:** Refresh / Content Opportunity Scoring. Not switching.

**Done looks like (this run):** two signal verdicts with bucket tables and `n` (staleness → MIXED, volume → MIXED; both flag-linked); one rule with a score, one reason code (`stale_visible_page`), one action (`refresh`); ranked CSV written from this notebook; twenty reviewed rows with "what would make it wrong"; no future-window or label-derived inputs in the score.
